# Project Example — 2015 U.S. Flight Delays

**Original:** Laura S. Bruckman, *Redesigns Practicum*  
**Python/Jupyter adaptation**

This notebook recreates the original R Markdown project example using Python. It is intentionally an **example of a developing project**, so several prompts remain for students to investigate rather than being fully resolved.

**Data source identified in the original material:** Kaggle, *2015 Flight Delays and Cancellations* (U.S. DOT/Bureau of Transportation Statistics data).

Expected local files:

```text
./data/flights.csv
./data/airports.csv
./data/airlines.csv
```

## Packages

The original example used the tidyverse/ggplot ecosystem. Here we use core Python data-science tools:

- `pandas` for tabular data,
- `numpy` for numerical work,
- `matplotlib` for visualization.

In [ ]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

DATA_DIR = Path("./data")

# Introduction

The U.S. Department of Transportation's Bureau of Transportation Statistics tracks the on-time performance of domestic flights operated by large air carriers.

Summary information includes the number of:

- on-time flights,
- delayed flights,
- canceled flights,
- diverted flights.

The example dataset contains 2015 flight-delay and cancellation data.

### Why is this important?

The data can be used to investigate:

- which airlines experience delays,
- which flights or routes are typically delayed,
- how flight volume varies,
- temporal patterns in flight operations.

**Project prompt:** Explain further why these questions matter and identify the specific question(s) your project will answer.

# Exploratory Data Analysis

## Explanation of the Dataset

There are three CSV files in the original Kaggle dataset.

### Read the three datasets

In [ ]:
flights_raw = pd.read_csv(DATA_DIR / "flights.csv")
airports_raw = pd.read_csv(DATA_DIR / "airports.csv")
airlines_raw = pd.read_csv(DATA_DIR / "airlines.csv")

print("flights:", flights_raw.shape)
print("airports:", airports_raw.shape)
print("airlines:", airlines_raw.shape)

### Look at the structure of the data

`DataFrame.info()` is a useful Python analogue to inspecting the structure of an R data frame.

In [ ]:
print("FLIGHTS")
flights_raw.info()

print("\nAIRPORTS")
airports_raw.info()

print("\nAIRLINES")
airlines_raw.info()

### Preview and summarize the datasets

The original R notebook used `skimr::skim()` and `summary()`. In pandas, `head()`, `info()`, and `describe(include="all")` provide similar first-pass information.

In [ ]:
display(flights_raw.head())
display(airports_raw.head())
display(airlines_raw.head())

In [ ]:
display(flights_raw.describe(include="all").T)
display(airports_raw.describe(include="all").T)
display(airlines_raw.describe(include="all").T)

### Missing-data overview

In [ ]:
def missing_summary(df):
    out = pd.DataFrame({
        "missing": df.isna().sum(),
        "percent_missing": 100 * df.isna().mean(),
        "dtype": df.dtypes.astype(str),
    })
    return out.sort_values("percent_missing", ascending=False)

print("FLIGHTS")
display(missing_summary(flights_raw).head(25))

print("AIRPORTS")
display(missing_summary(airports_raw))

print("AIRLINES")
display(missing_summary(airlines_raw))

### Visualizations for Exploratory Analysis

Questions to consider:

- Is the data consistent?
- Is it balanced or skewed?
- Are there categories with very few observations?
- Are there suspicious gaps or impossible values?

In [ ]:
def plot_counts(df, column, top=None, figsize=(9, 4), title=None):
    counts = df[column].value_counts(dropna=False)
    if top is not None:
        counts = counts.head(top)
    else:
        # Sort numeric/index-like categories naturally when possible.
        try:
            counts = counts.sort_index()
        except TypeError:
            pass

    ax = counts.plot(kind="bar", figsize=figsize)
    ax.set_xlabel(column)
    ax.set_ylabel("Count")
    ax.set_title(title or f"Count by {column}")
    plt.tight_layout()
    plt.show()

plot_counts(flights_raw, "MONTH")
plot_counts(flights_raw, "DAY")
plot_counts(flights_raw, "DAY_OF_WEEK")
plot_counts(flights_raw, "ORIGIN_AIRPORT", top=30,
            title="Top 30 origin airports by number of flights")
plot_counts(flights_raw, "AIRLINE")
plot_counts(airports_raw, "STATE", figsize=(12, 4))

## Data Cleaning

### Changing columns

The original example notes that the date columns need to be converted to a time-series-friendly form and that additional cleaning is required beyond what is shown here.

Questions to investigate:

- How many rows are dropped by the required-variable filter?
- Why are they dropped?
- Could dropping them bias the analysis?
- Are missing values concentrated in one airline, airport, month, or other subgroup?

In [ ]:
required = [
    "AIRLINE", "YEAR", "MONTH", "DAY",
    "ORIGIN_AIRPORT", "DEPARTURE_DELAY"
]

before_rows = len(flights_raw)

flights_df = flights_raw.dropna(subset=required).copy()

flights_df["FLIGHT_NUMBER"] = flights_df["FLIGHT_NUMBER"].astype("string")
flights_df["DAY_OF_WEEK"] = flights_df["DAY_OF_WEEK"].astype("Int64").astype("category")
flights_df["AIRLINE_SHORT"] = flights_df["AIRLINE"].astype("category")
flights_df["YMD"] = pd.to_datetime(
    flights_df[["YEAR", "MONTH", "DAY"]].rename(
        columns={"YEAR": "year", "MONTH": "month", "DAY": "day"}
    )
)
flights_df["is_delayed"] = (flights_df["DEPARTURE_DELAY"] >= 15).astype(int)

# Keep the short airline code under AIRLINE_SHORT rather than AIRLINE.
flights_df = flights_df.drop(columns="AIRLINE")

after_rows = len(flights_df)
print(f"Rows before cleaning: {before_rows:,}")
print(f"Rows after required-value filter: {after_rows:,}")
print(f"Rows dropped: {before_rows - after_rows:,}")

display(flights_df.head())

### Investigate the dropped rows

Cleaning is not just deleting missing values. Determine **why** values are missing and whether your choice could bias the result.

In [ ]:
dropped_mask = flights_raw[required].isna().any(axis=1)
dropped_rows = flights_raw.loc[dropped_mask].copy()

print("Number of dropped rows:", len(dropped_rows))
display(missing_summary(dropped_rows).head(20))

# Example: examine whether dropped rows are concentrated by airline/month.
if "AIRLINE" in dropped_rows.columns:
    display(dropped_rows["AIRLINE"].value_counts(dropna=False).head(20))
if "MONTH" in dropped_rows.columns:
    display(dropped_rows["MONTH"].value_counts(dropna=False).sort_index())

### Combine the multiple DataFrames

Both the airport and airline tables use an `IATA_CODE` field, but the meaning differs by table:

- in `airports.csv`, it identifies an airport;
- in `airlines.csv`, it identifies an airline.

Rename or explicitly specify merge keys so the meaning is clear.

**Questions:**

- Were any rows dropped by the merges?
- If so, why?
- Are there origin-airport codes without matching airport records?
- Are there airline codes without matching airline records?

In [ ]:
# Merge airport metadata onto origin-airport codes.
combined_flights = flights_df.merge(
    airports_raw,
    how="inner",
    left_on="ORIGIN_AIRPORT",
    right_on="IATA_CODE",
    validate="many_to_one"
)

print("Rows after airport merge:", len(combined_flights))

print("Flight airline codes:", sorted(flights_df["AIRLINE_SHORT"].astype(str).unique())[:20])
print("Airline table codes:", sorted(airlines_raw["IATA_CODE"].astype(str).unique())[:20])

# Rename the airline code so the merge key is explicit.
airlines_df = airlines_raw.rename(columns={"IATA_CODE": "AIRLINE_SHORT"}).copy()
airlines_df["AIRLINE_SHORT"] = airlines_df["AIRLINE_SHORT"].astype("category")

combined_flights = combined_flights.merge(
    airlines_df,
    how="inner",
    on="AIRLINE_SHORT",
    validate="many_to_one",
    suffixes=("_AIRPORT", "_AIRLINE")
)

print("Rows after airline merge:", len(combined_flights))
display(combined_flights.head())

### Check for unmatched keys

In [ ]:
airport_codes = set(airports_raw["IATA_CODE"].dropna().astype(str))
flight_airports = set(flights_df["ORIGIN_AIRPORT"].dropna().astype(str))
print("Origin-airport codes without airport metadata:")
print(sorted(flight_airports - airport_codes)[:50])

airline_codes = set(airlines_raw["IATA_CODE"].dropna().astype(str))
flight_airlines = set(flights_df["AIRLINE_SHORT"].dropna().astype(str))
print("\nAirline codes without airline metadata:")
print(sorted(flight_airlines - airline_codes))

### Make new variables and explore the data

The original example calculates the number of flights for each airline on each day and then merges that value back into the row-level flight table.

In [ ]:
flightnumber = (
    combined_flights
    .groupby(["AIRLINE_SHORT", "YMD"], observed=True)
    .size()
    .rename("nb_flights_airline")
    .reset_index()
)

combined_flights = combined_flights.merge(
    flightnumber,
    how="left",
    on=["AIRLINE_SHORT", "YMD"],
    validate="many_to_one"
)

top_airlines = (
    combined_flights
    .groupby("AIRLINE_SHORT", observed=True)
    .size()
    .rename("nb_flights")
    .sort_values(ascending=False)
    .reset_index()
)

display(top_airlines.head(15))

# Data Visualizations

## Evolution of Number of Flights

The original example plots the temporal evolution of daily flight counts for major U.S. airlines.

In [ ]:
top_codes = ["AA", "DL", "US", "EV", "WN"]
plot_df = (
    flightnumber[flightnumber["AIRLINE_SHORT"].astype(str).isin(top_codes)]
    .merge(
        airlines_df[["AIRLINE_SHORT", "AIRLINE"]],
        how="left",
        on="AIRLINE_SHORT",
        validate="many_to_one"
    )
    .sort_values("YMD")
)

fig, ax = plt.subplots(figsize=(12, 6))

for airline, group in plot_df.groupby("AIRLINE"):
    ax.plot(group["YMD"], group["nb_flights_airline"], label=airline)

ax.axvline(pd.Timestamp("2015-07-01"), linestyle="--", alpha=0.5)
ax.set_xlabel("Time")
ax.set_ylabel("Number of daily flights")
ax.set_title("Number of Daily Flights in 2015 for Selected Major Airlines")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
plt.xticks(rotation=25)
ax.legend(title="Airline", loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3)
fig.text(0.99, 0.01, "Source: publicly available U.S. DOT flight data", ha="right")
plt.tight_layout()
plt.show()

### How could this visualization be improved?

Ideas from the original project example:

- Decompose the signal to remove seasonality.
- Account for day-of-week effects.
- Improve legend placement or directly label the lines.
- Add an annotation for the merger between US Airways and American Airlines.
- Reduce date-label frequency.
- If seasonality is removed, consider thicker lines.
- Change the title to communicate the seasonal pattern.
- Annotate major holidays.
- Investigate October or other suspicious gaps and avoid connecting across missing periods without explanation.
- Investigate **where people are flying** in addition to airline-level volume.

## Other Visualizations and Questions

Possible next analyses:

- Flights per day.
- Delays per day, month, holiday, and day of week.
- Delays by time of day.
- Cancellation reasons.
- What does `WEATHER_DELAY` represent and when is it populated?
- City- or state-level information.
- Origin/destination route patterns.

In [ ]:
# Example starting point: departure-delay rate by month
monthly_delay = (
    combined_flights
    .groupby("MONTH", observed=True)["is_delayed"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "delay_rate", "count": "n_flights"})
)

monthly_delay["delay_rate_percent"] = 100 * monthly_delay["delay_rate"]
display(monthly_delay)

ax = monthly_delay["delay_rate_percent"].plot(kind="bar", figsize=(9, 4))
ax.set_xlabel("Month")
ax.set_ylabel("Delayed departures (%)")
ax.set_title("Departure-delay rate by month")
plt.tight_layout()
plt.show()

# Statistical Learning: Modeling & Prediction

The original example leaves this section open. A project should formulate a modeling question before selecting a model.

Examples:

- **Regression:** predict departure-delay minutes.
- **Classification:** predict whether a flight will be delayed by at least 15 minutes.

Before modeling:

1. Define the response variable.
2. Decide which predictors would actually be known at prediction time.
3. Avoid target leakage.
4. Split training/test data appropriately; for time-dependent questions, consider a chronological split.
5. Establish a simple baseline before fitting more complex models.

In [ ]:
# Example modeling imports — use only after defining a valid prediction question.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Build your feature set and train/test split here.
# Be careful not to include information that would only be available after departure.

# Conclusions

Summarize what the analysis established, what remains uncertain, and what additional data or modeling would be required to answer the original project question convincingly.